# AIC Port Pose — YOLOv11n-pose — University Cluster

Upload `aic_pose_model.zip` to the same directory as this notebook on JupyterHub, then run all cells top-to-bottom.

**Dataset:** 2 classes — `SC_SKL` (SC port), `SPF_SKL` (SFP port) — 17 keypoints each  
**Model:** `yolo11n-pose` fine-tuned from COCO pose pretrained weights  
**Output:** `aic_output.zip` — download from JupyterHub file browser after training

In [ ]:
# ── GPU check ─────────────────────────────────────────────────────────────────
import subprocess
r = subprocess.run(
    'nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader',
    shell=True, capture_output=True, text=True
)
if r.returncode == 0:
    print(f'GPU: {r.stdout.strip()}')
else:
    print('⚠  No GPU detected — check your cluster job allocation')

In [ ]:
# ── Install dependencies (cluster-safe: --user avoids permission errors) ──────
import subprocess, sys, site

subprocess.check_call([
    sys.executable, '-m', 'pip', 'install',
    'ultralytics', 'Pillow', '-q', '--user'
])

# Append user site-packages so ultralytics is importable without restarting kernel
user_site = site.getusersitepackages()
if user_site not in sys.path:
    sys.path.append(user_site)

import ultralytics, torch
print(f'Ultralytics {ultralytics.__version__}')
print(f'PyTorch {torch.__version__}  CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ── Extract aic_pose_model.zip ────────────────────────────────────────────────
# Upload aic_pose_model.zip to the same directory as this notebook before running.
import os, zipfile, shutil
from pathlib import Path

NOTEBOOK_DIR = Path(os.getcwd()).resolve()
ZIP_PATH     = NOTEBOOK_DIR / 'aic_pose_model.zip'
ROOT         = NOTEBOOK_DIR / 'aic_pose_model'
OUTPUT_DIR   = NOTEBOOK_DIR / 'aic_output'
OUTPUT_DIR.mkdir(exist_ok=True)

if not ZIP_PATH.exists():
    raise FileNotFoundError(
        f'{ZIP_PATH} not found.\n'
        f'Upload aic_pose_model.zip to {NOTEBOOK_DIR} then re-run this cell.'
    )

# Always re-extract to guarantee a clean state (safe — zip is the source of truth)
if ROOT.exists():
    shutil.rmtree(ROOT)
print(f'Extracting {ZIP_PATH} ...')
with zipfile.ZipFile(ZIP_PATH) as zf:
    zf.extractall(NOTEBOOK_DIR)
print(f'Extracted to {ROOT}')

# Verify expected structure
for name in ('data', 'labels', 'data.yaml'):
    p = ROOT / name
    status = '✓' if p.exists() else '✗ MISSING'
    print(f'  {name:<20} {status}')

print(f'\nOutput dir: {OUTPUT_DIR}')

In [ ]:
# ── Prepare dataset ───────────────────────────────────────────────────────────
from pathlib import Path
from PIL import Image, ImageFile
from collections import defaultdict, Counter
import shutil, yaml

ImageFile.LOAD_TRUNCATED_IMAGES = True

DATA_DIR     = ROOT / 'data'
LABELS_DIR   = ROOT / 'labels'
IMAGES_TRAIN = DATA_DIR / 'images' / 'train'
TRAIN_TXT    = ROOT / 'train.txt'
VAL_TXT      = ROOT / 'val.txt'
VAL_FRACTION = 0.2
SUPPORTED    = {'.ppm', '.png', '.jpg', '.jpeg'}

# ── Diagnose structure ────────────────────────────────────────────────────────
print(f'ROOT: {ROOT}')
print(f'ROOT contents:')
for p in sorted(ROOT.iterdir()):
    n = sum(1 for _ in p.rglob('*') if _.is_file()) if p.is_dir() else ''
    print(f'  [dir]  {p.name}  ({n} files)' if p.is_dir() else f'  [file] {p.name}')

print(f'\ndata/ contents:')
if DATA_DIR.exists():
    for p in sorted(DATA_DIR.iterdir()):
        n = sum(1 for _ in p.rglob('*') if _.is_file()) if p.is_dir() else ''
        print(f'  [dir]  {p.name}  ({n} files)' if p.is_dir() else f'  [file] {p.name}')
else:
    print('  (data/ does not exist!)')

# ── Delete stale .cache files (created on another machine — paths are wrong) ──
removed = [f for f in ROOT.rglob('*.cache')]
for f in removed:
    f.unlink()
if removed:
    print(f'\nRemoved {len(removed)} stale .cache files')

# ── Find source images ────────────────────────────────────────────────────────
subject_dirs = sorted(
    p for p in DATA_DIR.iterdir()
    if p.is_dir() and p.name not in {'images', 'labels'}
) if DATA_DIR.exists() else []

if subject_dirs:
    print(f'\nCase A — raw annotator dirs: {[p.name for p in subject_dirs]}')
    if IMAGES_TRAIN.exists():
        shutil.rmtree(IMAGES_TRAIN)
    IMAGES_TRAIN.mkdir(parents=True)
    converted = []
    for subject_dir in subject_dirs:
        for src in sorted(subject_dir.rglob('*')):
            if not src.is_file() or src.suffix.lower() not in SUPPORTED:
                continue
            rel = src.relative_to(DATA_DIR).with_suffix('.png')
            dst = IMAGES_TRAIN / rel
            dst.parent.mkdir(parents=True, exist_ok=True)
            with Image.open(src) as img:
                img.save(dst, format='PNG')
            converted.append(dst)
    print(f'Converted {len(converted)} images → data/images/train/')

elif IMAGES_TRAIN.exists() and any(IMAGES_TRAIN.rglob('*.png')):
    converted = sorted(IMAGES_TRAIN.rglob('*.png'))
    print(f'\nCase B — using {len(converted)} pre-converted PNGs from data/images/train/')

else:
    raise RuntimeError(
        f'No source images found.\n'
        f'Expected either:\n'
        f'  A) raw annotator dirs at   {DATA_DIR}/Abi/, ...\n'
        f'  B) converted PNGs already at {IMAGES_TRAIN}\n'
        f'Check your zip: run  unzip -l aic_pose_model.zip | head -40  locally.'
    )

print(f'Labels dir: {LABELS_DIR}  exists={LABELS_DIR.exists()}')

# ── Copy labels into data/labels/train/ ───────────────────────────────────────
# YOLO replaces 'images' with 'labels' in the image path to find labels:
#   data/images/train/Abi/x.png  →  data/labels/train/Abi/x.txt
data_labels = DATA_DIR / 'labels'
if data_labels.is_symlink() or data_labels.is_file():
    data_labels.unlink()
elif data_labels.exists():
    shutil.rmtree(data_labels)

data_labels_train = data_labels / 'train'
data_labels_train.mkdir(parents=True)

src_labels_train = LABELS_DIR / 'train'
if not src_labels_train.exists():
    raise RuntimeError(
        f'Labels directory not found: {src_labels_train}\n'
        f'Make sure labels/ is included in your zip.'
    )

for item in sorted(src_labels_train.iterdir()):
    dst = data_labels_train / item.name
    if item.is_dir():
        shutil.copytree(str(item), str(dst))
    else:
        shutil.copy2(str(item), str(dst))

n_lbl = sum(1 for _ in data_labels_train.rglob('*.txt'))
print(f'Copied {n_lbl} label files → data/labels/train/')

# ── Validate image ↔ label coverage ──────────────────────────────────────────
missing_labels = []
for dst in converted:
    rel = dst.relative_to(IMAGES_TRAIN)
    lbl = data_labels_train / rel.with_suffix('.txt')
    if not lbl.exists():
        missing_labels.append(str(rel))

if missing_labels:
    print(f'\n⚠  {len(missing_labels)} images are missing label files (first 10):')
    for m in missing_labels[:10]:
        print(f'   {m}')
    raise RuntimeError(
        f'{len(missing_labels)} images have no matching label files.\n'
        f'Check that labels/train/ mirrors the image folder structure.'
    )
print(f'Label coverage: ✓ all {len(converted)} images have labels')

# ── Stratified 80/20 train/val split ─────────────────────────────────────────
by_class = defaultdict(list)
class_counts = Counter()
kp_counts    = Counter()

for dst in converted:
    rel = dst.relative_to(IMAGES_TRAIN)
    lbl = data_labels_train / rel.with_suffix('.txt')
    for line in lbl.read_text().splitlines():
        v = line.strip().split()
        if not v: continue
        cls_id = int(v[0])
        class_counts[cls_id] += 1
        kp_counts[(len(v) - 5) // 3] += 1
        # Use absolute paths — no ambiguity regardless of working directory
        by_class[cls_id].append(str(dst))
        break

train_lines, val_lines = [], []
for cls_id, lines in sorted(by_class.items()):
    lines  = sorted(set(lines))
    n_val  = max(1, round(len(lines) * VAL_FRACTION)) if len(lines) > 1 else 0
    step   = max(1, len(lines) // n_val) if n_val else 1
    val_set = set(lines[::step][:n_val]) if n_val else set()
    val_lines.extend(sorted(val_set))
    train_lines.extend(l for l in lines if l not in val_set)

TRAIN_TXT.write_text('\n'.join(sorted(train_lines)) + '\n')
VAL_TXT.write_text(  '\n'.join(sorted(val_lines))   + '\n')

print(f'Train: {len(train_lines)}  Val: {len(val_lines)}')
print(f'Classes: { {k: v for k,v in sorted(class_counts.items())} }')
print(f'Keypoints per object: {dict(kp_counts)}')

# ── Patch data.yaml with absolute path ────────────────────────────────────────
with open(ROOT / 'data.yaml') as f:
    data_cfg = yaml.safe_load(f)

data_cfg['path'] = str(ROOT)

patched_yaml = OUTPUT_DIR / 'data.yaml'
with open(patched_yaml, 'w') as f:
    yaml.dump(data_cfg, f, default_flow_style=False)

print(f'\ndata.yaml patched → {patched_yaml}')
print(yaml.dump(data_cfg, default_flow_style=False))

In [ ]:
# ── Validate dataset ──────────────────────────────────────────────────────────
from collections import Counter
from pathlib import Path

train_lines = [l.strip() for l in (ROOT / 'train.txt').read_text().splitlines() if l.strip()]
val_lines   = [l.strip() for l in (ROOT / 'val.txt').read_text().splitlines()   if l.strip()]

class_counts = Counter()
kp_counts    = Counter()
missing      = []

for img_line in train_lines + val_lines:
    img_path = Path(img_line)   # absolute path
    # label is found by replacing /data/images/train/ with /data/labels/train/
    lbl_path = Path(str(img_path).replace('/data/images/train/', '/data/labels/train/')).with_suffix('.txt')
    if not lbl_path.exists():
        missing.append(str(lbl_path))
        continue
    for line in lbl_path.read_text().splitlines():
        v = line.strip().split()
        if not v: continue
        class_counts[int(v[0])] += 1
        kp_counts[(len(v) - 5) // 3] += 1

class_names = data_cfg.get('names', {})
print(f'Train images : {len(train_lines)}')
print(f'Val images   : {len(val_lines)}')
print(f'Missing labels: {len(missing)}')
print(f'Class counts  : { {class_names.get(k,k): v for k,v in sorted(class_counts.items())} }')
print(f'KP counts     : {dict(kp_counts)}')
if missing:
    print('⚠  Missing label files:')
    for m in missing[:5]:
        print(f'   {m}')

In [ ]:
# ── Visualise sample annotations ──────────────────────────────────────────────
import random, cv2
import numpy as np
import matplotlib.pyplot as plt

N_SHOW = 8
samples = random.sample(train_lines, min(N_SHOW, len(train_lines)))
fig, axes = plt.subplots(2, 4, figsize=(22, 9))
for ax in axes.flatten():
    ax.axis('off')

KP_COLOR = (255, 50, 50)

for ax, img_line in zip(axes.flatten(), samples):
    img_path = Path(img_line)   # absolute path
    lbl_path = Path(str(img_path).replace('/data/images/train/', '/data/labels/train/')).with_suffix('.txt')

    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    H, W = img.shape[:2]

    for raw in lbl_path.read_text().splitlines():
        vals = list(map(float, raw.strip().split()))
        if not vals: continue
        cls_id = int(vals[0])
        cx, cy, bw, bh = vals[1]*W, vals[2]*H, vals[3]*W, vals[4]*H
        x0, y0 = int(cx - bw/2), int(cy - bh/2)
        color = (50, 200, 50) if cls_id == 0 else (50, 100, 220)
        cv2.rectangle(img, (x0,y0), (x0+int(bw),y0+int(bh)), color, 2)
        name = class_names.get(cls_id, str(cls_id))
        cv2.putText(img, name, (x0, max(y0-5,10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
        for i in range(0, len(vals)-5, 3):
            base = 5 + i
            u, v, vis = vals[base]*W, vals[base+1]*H, vals[base+2]
            if vis > 0:
                cv2.circle(img, (int(u),int(v)), 4, KP_COLOR, -1)

    ax.imshow(img)
    ax.set_title(img_path.parent.name, fontsize=8)
    ax.axis('on'); ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('Sample training annotations — verify keypoints before training', fontsize=12)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'annotations_preview.png', dpi=80)
plt.show()

In [ ]:
# ── Training hyperparameters ──────────────────────────────────────────────────
MODEL_NAME = 'yolo11n-pose'   # nano — fast; swap to yolo11s-pose if you have time
EPOCHS     = 150
IMGSZ      = 640              # 640 is the YOLO standard; use 1024 only if GPU has >16 GB
BATCH      = 16               # reduce to 8 if you get OOM
LR0        = 0.001
LRF        = 0.01
PATIENCE   = 50
WORKERS    = 4
DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Model   : {MODEL_NAME}')
print(f'Device  : {DEVICE}')
print(f'Epochs  : {EPOCHS}')
print(f'ImgSz   : {IMGSZ}')
print(f'Batch   : {BATCH}')

In [ ]:
# ── Train ─────────────────────────────────────────────────────────────────────
from ultralytics import YOLO

model   = YOLO(f'{MODEL_NAME}.pt')
results = model.train(
    data       = str(patched_yaml),
    epochs     = EPOCHS,
    imgsz      = IMGSZ,
    batch      = BATCH,
    workers    = WORKERS,
    device     = DEVICE,
    lr0        = LR0,
    lrf        = LRF,
    patience   = PATIENCE,
    project    = str(OUTPUT_DIR),
    name       = 'port_pose',
    exist_ok   = True,
    # Augmentation — conservative for small industrial dataset
    hsv_h      = 0.015,
    hsv_s      = 0.4,
    hsv_v      = 0.4,
    degrees    = 5.0,
    translate  = 0.05,
    scale      = 0.3,
    fliplr     = 0.0,   # ports have fixed orientation
    flipud     = 0.0,
    mosaic     = 0.5,
    verbose    = True,
)

best_pt = results.save_dir / 'weights' / 'best.pt'
print(f'\n✓ Training complete')
print(f'  Best weights: {best_pt}')

In [ ]:
# ── Validate ──────────────────────────────────────────────────────────────────
best_model  = YOLO(str(best_pt))
val_metrics = best_model.val(data=str(patched_yaml), device=DEVICE, verbose=True)
print(f'mAP50-95 (pose): {val_metrics.pose.map:.4f}')
print(f'mAP50    (pose): {val_metrics.pose.map50:.4f}')

In [ ]:
# ── Export ONNX + package everything for download ─────────────────────────────
import shutil, zipfile as _zf

# ONNX export (for CPU inference on the robot)
best_model.export(format='onnx', imgsz=IMGSZ, opset=12)
onnx_path = best_pt.with_suffix('.onnx')

# Copy key files to OUTPUT_DIR root
shutil.copy(best_pt, OUTPUT_DIR / 'best.pt')
# patched_yaml is already at OUTPUT_DIR/data.yaml — no copy needed
if onnx_path.exists():
    shutil.copy(onnx_path, OUTPUT_DIR / 'best.onnx')

# Also copy camera_info.json if present
for ci in [ROOT / 'camera_info.json', NOTEBOOK_DIR / 'camera_info.json']:
    if ci.exists():
        shutil.copy(ci, OUTPUT_DIR / 'camera_info.json')
        break

# Zip the output folder
zip_path = NOTEBOOK_DIR / 'aic_output.zip'
with _zf.ZipFile(zip_path, 'w', _zf.ZIP_DEFLATED) as zf:
    for ext in ('*.pt', '*.onnx', '*.yaml', '*.json', '*.png'):
        for src in OUTPUT_DIR.glob(ext):
            zf.write(src, src.name)

print(f'\n✓ Download this file from JupyterHub:')
print(f'   {zip_path}  ({zip_path.stat().st_size/1024/1024:.1f} MB)')
print(f'\nContents:')
for p in sorted(OUTPUT_DIR.glob('*.*')):
    print(f'  {p.name:<30} {p.stat().st_size/1024/1024:.2f} MB')

## Inference — Port Detection Visualizer

Run this cell to verify the trained model on any image.  
Green = **SC_SKL** (class 0) · Blue = **SPF_SKL** (class 1)

Two modes:
- Set `IMAGE_PATH` to a local file path, **or**
- Leave `IMAGE_PATH = None` → an upload button appears (works on JupyterHub)

In [ ]:
"""
Port detection inference visualizer.

Usage
-----
Option A — file path (fastest):
    Set IMAGE_PATH to any PNG / JPG / PPM file, then run the cell.

Option B — upload button (JupyterHub):
    Leave IMAGE_PATH = None.  An upload widget appears; pick a file and the
    inference runs automatically.

Output: annotated image with bounding boxes, keypoints, insertion-axis arrow,
        and a text table showing confidence + visible keypoint count.
"""
import io
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from PIL import Image as PILImage
from ultralytics import YOLO

# ── Config — edit these ───────────────────────────────────────────────────────
IMAGE_PATH  = None    # e.g. Path("aic_pose_model/data/images/train/Abi/sc/sc_0001.png")
CONF_THRESH = 0.25    # lower to see weak detections; raise to suppress noise
KP_CONF_MIN = 0.30    # minimum per-keypoint confidence to display

# ── Locate best.pt ────────────────────────────────────────────────────────────
_candidates = [
    OUTPUT_DIR / "best.pt",
    OUTPUT_DIR / "port_pose" / "weights" / "best.pt",
]
_model_path = next((p for p in _candidates if p.is_file()), None)
if _model_path is None:
    raise FileNotFoundError(
        "best.pt not found. Run the Train and Export cells first, "
        f"or place best.pt in {OUTPUT_DIR}."
    )

CLASS_NAMES  = {0: "SC_SKL", 1: "SPF_SKL"}
CLASS_COLORS = {0: "#00e676", 1: "#448aff"}   # green / blue

print(f"Model : {_model_path}")
_model = YOLO(str(_model_path))
print(f"Ready — classes: {CLASS_NAMES}")


# ── Inference + visualisation ─────────────────────────────────────────────────
def run_inference(pil_img: PILImage.Image, title: str = "") -> None:
    img_np  = np.array(pil_img.convert("RGB"))
    results = _model(img_np, conf=CONF_THRESH, verbose=False)

    fig, ax = plt.subplots(figsize=(11, 8))
    ax.imshow(img_np)
    ax.set_title(
        f"YOLO-pose  conf≥{CONF_THRESH}"
        + (f"  |  {title}" if title else ""),
        fontsize=11,
    )

    total_dets = 0
    for r in results:
        if r.boxes is None:
            continue
        boxes   = r.boxes.xywh.cpu().numpy()      # cx cy w h  (pixels)
        confs   = r.boxes.conf.cpu().numpy()
        classes = r.boxes.cls.cpu().numpy().astype(int)
        kp_xy   = r.keypoints.xy.cpu().numpy()    # (N,17,2)
        kp_conf = (
            r.keypoints.conf.cpu().numpy()
            if r.keypoints.conf is not None
            else np.ones((len(boxes), 17), dtype=np.float32)
        )

        for i, (box, conf, cls_id) in enumerate(zip(boxes, confs, classes)):
            cx, cy, bw, bh = box
            color    = CLASS_COLORS.get(cls_id, "#ff1744")
            cls_name = CLASS_NAMES.get(cls_id, f"cls{cls_id}")
            total_dets += 1

            # Bounding box
            rect = mpatches.FancyBboxPatch(
                (cx - bw / 2, cy - bh / 2), bw, bh,
                boxstyle="round,pad=2", linewidth=2,
                edgecolor=color, facecolor="none",
            )
            ax.add_patch(rect)

            # Label
            ax.text(
                cx - bw / 2, cy - bh / 2 - 6,
                f"{cls_name}  {conf:.2f}",
                color="white", fontsize=9, fontweight="bold",
                bbox=dict(facecolor=color, alpha=0.80, pad=2, linewidth=0),
            )

            # Keypoints
            kps    = kp_xy[i]       # (17,2)
            kconfs = kp_conf[i]     # (17,)
            vis    = kconfs > KP_CONF_MIN
            if vis.any():
                ax.scatter(
                    kps[vis, 0], kps[vis, 1],
                    c=color, s=35, zorder=5, linewidths=0.6,
                    edgecolors="white",
                )
                # Arrow pointing "up" from centroid (rough insertion direction hint)
                cx_kp = kps[vis, 0].mean()
                cy_kp = kps[vis, 1].mean()
                ax.annotate(
                    "", xy=(cx_kp, cy_kp - 22), xytext=(cx_kp, cy_kp),
                    arrowprops=dict(arrowstyle="-|>", color=color, lw=2.0),
                    zorder=6,
                )

            # Compact keypoint confidence strip below box
            kp_text = "  ".join(
                f"kp{j}:{kc:.2f}"
                for j, kc in enumerate(kconfs)
                if kc > KP_CONF_MIN
            )
            if kp_text:
                ax.text(
                    cx - bw / 2, cy + bh / 2 + 9,
                    kp_text, color=color, fontsize=5.5,
                )

    legend_handles = [
        mpatches.Patch(color=CLASS_COLORS[0], label="SC_SKL (0)"),
        mpatches.Patch(color=CLASS_COLORS[1], label="SPF_SKL (1)"),
    ]
    ax.legend(handles=legend_handles, loc="upper right", fontsize=9)
    ax.axis("off")
    plt.tight_layout()
    plt.show()

    # ── Text summary ──────────────────────────────────────────────────────────
    if total_dets == 0:
        print(f"No detections above conf={CONF_THRESH}")
        return
    print(f"\n{'#':>3}  {'Class':<10}  {'Conf':>5}  {'bbox cx,cy,w,h (px)':>30}  visible_kp")
    for r in results:
        if r.boxes is None:
            continue
        for i in range(len(r.boxes)):
            cls_id   = int(r.boxes.cls[i])
            conf_val = float(r.boxes.conf[i])
            b        = r.boxes.xywh[i].cpu().numpy()
            n_vis    = (
                int((r.keypoints.conf[i] > KP_CONF_MIN).sum())
                if r.keypoints.conf is not None else "?"
            )
            print(
                f"{i:>3}  {CLASS_NAMES.get(cls_id, cls_id):<10}  "
                f"{conf_val:>5.3f}  "
                f"  cx={b[0]:>5.0f}  cy={b[1]:>5.0f}  "
                f"w={b[2]:>4.0f}  h={b[3]:>4.0f}    "
                f"{n_vis}/17"
            )


# ── Run ───────────────────────────────────────────────────────────────────────
if IMAGE_PATH is not None:
    _img = PILImage.open(IMAGE_PATH)
    print(f"Loaded: {IMAGE_PATH}  ({_img.size[0]}×{_img.size[1]})")
    run_inference(_img, title=Path(IMAGE_PATH).name)

else:
    try:
        import ipywidgets as widgets
        from IPython.display import display

        _uploader = widgets.FileUpload(
            accept="image/*", multiple=False,
            description="Upload image",
            layout=widgets.Layout(width="220px"),
        )
        display(_uploader)

        def _on_upload(change):
            if not _uploader.value:
                return
            item     = list(_uploader.value.values())[0]
            fname    = item["metadata"]["name"]
            content  = item["content"]
            _img     = PILImage.open(io.BytesIO(content))
            print(f"\nUploaded: {fname}  ({_img.size[0]}×{_img.size[1]})")
            run_inference(_img, title=fname)

        _uploader.observe(_on_upload, names="value")
        print("← Pick an image with the button.  "
              "Or set IMAGE_PATH = Path('...') and re-run the cell.")

    except ImportError:
        print("ipywidgets not available.  Set IMAGE_PATH to a file path and re-run.")